# Clasificador neuronal para AG News

Este notebook implementa un clasificador de noticias usando redes neuronales en PyTorch y el dataset `ag_news`. Toma como base la idea del ejemplo `Neural_Networks_for_sentiment_analysis.py`, pero adapta el problema de clasificación binaria de sentimiento a clasificación multiclase.

AG News tiene 4 clases:

- `World`
- `Sports`
- `Business`
- `Sci/Tech`

Relación con el tema visto: el flujo completo usa tokenización, vocabulario, embeddings, una red feed-forward, función ReLU en capas ocultas, logits de salida, `CrossEntropyLoss`, backpropagation y gradient descent.

## 1. Instalación e imports

En Colab normalmente ya existe PyTorch, pero instalamos `datasets` para descargar AG News desde Hugging Face.

En esta sección se cargan las librerías principales:

- `torch`: construcción y entrenamiento de la red neuronal.
- `datasets`: descarga del dataset.
- `Counter`: construcción del vocabulario.
- `numpy`: cálculos de métricas.
- `re`: tokenización simple basada en expresiones regulares.

In [ ]:
!pip -q install datasets

import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)
np.random.seed(42)

## 2. Carga del dataset AG News

A diferencia del ejemplo de sentimiento IMDB, aquí no hay solo dos clases. El modelo debe elegir entre varias categorías, por eso el diseño cambia:

- En IMDB se usa una salida con `Sigmoid` para predecir positivo/negativo.
- En AG News usamos 4 salidas, una por clase.
- Durante entrenamiento usamos `CrossEntropyLoss`, que combina internamente `LogSoftmax` y pérdida de entropía cruzada.

Para que el notebook sea rápido en Colab, se puede entrenar con una muestra del dataset. Si quieres usar todo el entrenamiento, cambia `TRAIN_LIMIT = None`.

In [ ]:
dataset = load_dataset("ag_news")

label_names = dataset["train"].features["label"].names
num_classes = len(label_names)

TRAIN_LIMIT = 40000
TEST_LIMIT = None

train_data = dataset["train"]
test_data = dataset["test"]

if TRAIN_LIMIT is not None:
    train_data = train_data.shuffle(seed=42).select(range(TRAIN_LIMIT))

if TEST_LIMIT is not None:
    test_data = test_data.select(range(TEST_LIMIT))

print(dataset)
print("Clases:", label_names)
print("Ejemplo:")
print(train_data[0])

## 3. Tokenización y vocabulario

Una red neuronal no procesa texto directamente. Primero convertimos texto a tokens y luego tokens a IDs numéricos.

Este paso corresponde a la preparación del input vista en PLN:

```text
texto crudo -> tokens -> IDs -> embeddings
```

Usamos dos tokens especiales:

- `<PAD>`: relleno para que todos los textos tengan la misma longitud.
- `<UNK>`: palabra desconocida, no encontrada en el vocabulario.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text.lower())

VOCAB_SIZE = 30000
MAX_LENGTH = 80

counter = Counter()
for example in train_data:
    counter.update(tokenize(example["text"]))

word2idx = {"<PAD>": 0, "<UNK>": 1}
idx2word = {0: "<PAD>", 1: "<UNK>"}

for idx, (word, _) in enumerate(counter.most_common(VOCAB_SIZE - 2), start=2):
    word2idx[word] = idx
    idx2word[idx] = word

def encode(text):
    token_ids = [word2idx.get(token, word2idx["<UNK>"]) for token in tokenize(text)]
    token_ids = token_ids[:MAX_LENGTH]
    padding = [word2idx["<PAD>"]] * (MAX_LENGTH - len(token_ids))
    return token_ids + padding

print("Tamaño del vocabulario:", len(word2idx))
print("Texto original:", train_data[0]["text"])
print("Texto codificado:", encode(train_data[0]["text"])[:20])

## 4. Dataset y DataLoader

Creamos una clase compatible con PyTorch para entregar pares `(x, y)`:

- `x`: secuencia de IDs de palabras.
- `y`: clase correcta de la noticia.

`DataLoader` agrupa ejemplos en batches. Esto hace más eficiente el entrenamiento y permite aprovechar GPU si está disponible.

In [ ]:
class AGNewsDataset(Dataset):
    def __init__(self, split):
        self.data = split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        x = torch.tensor(encode(text), dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        return x, y

BATCH_SIZE = 256

train_dataset = AGNewsDataset(train_data)
test_dataset = AGNewsDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

batch_x, batch_y = next(iter(train_loader))
print("Forma de x:", batch_x.shape)
print("Forma de y:", batch_y.shape)

## 5. Modelo neuronal

La arquitectura se basa en el ejemplo de sentimiento:

```text
IDs -> Embedding -> Promedio global -> Capas densas -> Logits
```

Diferencias importantes contra clasificación binaria:

- La salida tiene `num_classes = 4` neuronas.
- No aplicamos `Sigmoid` al final.
- El modelo devuelve `logits`, es decir, puntuaciones crudas.
- `CrossEntropyLoss` se encarga de comparar esos logits contra la clase real.

Relación con el tema visto:

- `Embedding`: representación densa de palabras.
- `ReLU`: función de activación para introducir no linealidad.
- `Dropout`: regularización para reducir overfitting.
- `Linear`: capas feed-forward que aprenden combinaciones de características.
- `Logits`: salidas antes de Softmax.

In [ ]:
class AGNewsNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )
        self.fc1 = nn.Linear(embedding_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)       # (batch, seq_len, embedding_dim)
        x = x.mean(dim=1)           # (batch, embedding_dim)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        logits = self.fc3(x)        # (batch, num_classes)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

EMBEDDING_DIM = 100

model = AGNewsNet(
    vocab_size=len(word2idx),
    embedding_dim=EMBEDDING_DIM,
    num_classes=num_classes,
).to(device)

print(model)

## 6. Entrenamiento

El entrenamiento sigue las ideas de forward propagation, loss, backpropagation y gradient descent:

1. `outputs = model(inputs)`: forward propagation.
2. `loss = criterion(outputs, labels)`: cálculo de pérdida.
3. `loss.backward()`: backpropagation.
4. `optimizer.step()`: actualización de pesos.

Usamos `CrossEntropyLoss` porque es clasificación multiclase. No se debe usar `BCELoss` aquí, ya que esa pérdida corresponde al caso binario del ejemplo IMDB.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")

## 7. Evaluación

En clasificación multiclase, la matriz de confusión muestra cuántos ejemplos de cada clase fueron clasificados como cada posible clase.

También calculamos precisión, recall y F1 por clase:

- `Precision`: de lo que el modelo predijo como una clase, cuánto fue correcto.
- `Recall`: de lo que realmente pertenecía a una clase, cuánto encontró el modelo.
- `F1`: balance entre precisión y recall.

In [ ]:
def evaluate(model, loader, device, num_classes):
    model.eval()
    confusion = np.zeros((num_classes, num_classes), dtype=int)

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits = model(inputs)
            predictions = torch.argmax(logits, dim=1)

            for gold, pred in zip(labels.cpu().numpy(), predictions.cpu().numpy()):
                confusion[gold, pred] += 1

    accuracy = np.trace(confusion) / np.sum(confusion)
    return confusion, accuracy

confusion, test_accuracy = evaluate(model, test_loader, device, num_classes)

print("Accuracy en test:", round(test_accuracy, 4))
print("\nMatriz de confusión")
print("Filas = clase real | Columnas = clase predicha")
print(confusion)

print("\nMétricas por clase")
for i, class_name in enumerate(label_names):
    tp = confusion[i, i]
    fp = confusion[:, i].sum() - tp
    fn = confusion[i, :].sum() - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"{class_name:8s} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

## 8. Predicción con noticias nuevas

Para interpretar la salida, aplicamos `Softmax` solo en inferencia. Durante entrenamiento no lo aplicamos manualmente porque `CrossEntropyLoss` ya trabaja con logits.

La clase con mayor probabilidad es la predicción final.

In [ ]:
def predict_text(model, text, device):
    model.eval()
    x = torch.tensor(encode(text), dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
        predicted_idx = int(np.argmax(probabilities))

    return label_names[predicted_idx], probabilities

examples = [
    "The government announced new international trade measures after talks with European leaders.",
    "The team won the championship after scoring two goals in the final minutes.",
    "The company reported higher quarterly revenue as stock markets reacted positively.",
    "Researchers introduced a new chip design for faster artificial intelligence computing.",
]

for text in examples:
    predicted_class, probabilities = predict_text(model, text, device)
    print("Texto:", text)
    print("Predicción:", predicted_class)
    print("Probabilidades:", {label_names[i]: round(float(p), 4) for i, p in enumerate(probabilities)})
    print("-" * 90)

## 9. Relación directa con el tema de redes neuronales para PLN

Este notebook conecta con los conceptos principales del tema:

| Concepto | Dónde aparece en el notebook |
|---|---|
| Tokenización | Conversión de texto en palabras/tokens |
| Vocabulario | Mapeo `palabra -> ID` |
| Embeddings | `nn.Embedding`, convierte IDs en vectores densos |
| Forward propagation | `logits = model(inputs)` |
| ReLU | Activación de capas ocultas |
| Dropout | Regularización contra overfitting |
| Softmax | Interpretación final como probabilidades |
| Cross-Entropy | Pérdida para clasificación multiclase |
| Backpropagation | `loss.backward()` |
| Gradient descent | `optimizer.step()` con Adam |

La principal diferencia con el ejemplo de sentimiento es que ahora no resolvemos un problema binario, sino multiclase. Por eso se reemplaza `Sigmoid + BCELoss` por `logits + CrossEntropyLoss`.